In [30]:
import numpy as np
import sys
from keras.datasets import mnist

In [31]:
# Functions of non-linear activations
def f_sigmoid(X, deriv=False):
    if not deriv:
        return 1 / (1 + np.exp(-X))
    else:
        return f_sigmoid(X) * (1 - f_sigmoid(X))


def f_softmax(X):
    Z = np.sum(np.exp(X), axis=1)
    Z = Z.reshape(Z.shape[0], 1)
    return np.exp(X) / Z

# ReLU as output, doesn't need deriv but in case one wants it as an activator function
def f_ReLU(X, deriv=False):
    if not deriv: 
        return np.maximum(0, X)
    elif deriv: 
        # if X is smaller to 0 it returns 0 otherwise 1 
        return (X > 0).astype(float)


In [32]:
def exit_with_err(err_str):
    print("fatal error", err_str)
    #print >> sys.stderr, err_str
    sys.exit(1)

In [33]:
# Functionality of a single hidden layer
class Layer:
    def __init__(self, size, batch_size, is_input=False, is_output=False, activation=f_sigmoid):
        self.is_input = is_input
        self.is_output = is_output

        # Z is the matrix that holds output values
        self.Z = np.zeros((batch_size, size[0]))
        # The activation function is an externally defined function (with a derivative) that is stored here
        self.activation = activation

        # W is the outgoing weight matrix for this layer
        self.W = None
        # S is the matrix that holds the inputs to this layer
        self.S = None
        # D is the matrix that holds the deltas for this layer
        self.D = None
        # Fp is the matrix that holds the derivatives of the activation function
        self.Fp = None

        if not is_input:
            self.S = np.zeros((batch_size, size[0]))
            self.D = np.zeros((batch_size, size[0]))

        if not is_output:
            self.W = np.random.normal(size=size, scale=1E-4)

        if not is_input and not is_output:
            self.Fp = np.zeros((size[0], batch_size))

    def forward_propagate(self):
        if self.is_input:
            return self.Z.dot(self.W)

        self.Z = self.activation(self.S)
        if self.is_output:
            return self.Z
        else:
            # For hidden layers, we add the bias values here
            self.Z = np.append(self.Z, np.ones((self.Z.shape[0], 1)), axis=1)
            self.Fp = self.activation(self.S, deriv=True).T
            return self.Z.dot(self.W)


In [36]:
class MultiLayerPerceptron:
    def __init__(self, layer_config, batch_size=100):
        self.layers = []
        self.num_layers = len(layer_config)
        self.minibatch_size = batch_size

        for i in range(self.num_layers-1):
            if i == 0:
                print(f"Initializing input layer with size {layer_config[i]}.")
                # Here, we add an additional unit at the input for the bias weight.
                self.layers.append(Layer([layer_config[i]+1, layer_config[i+1]],
                                         batch_size,
                                         is_input=True))
            else:
                print(f"Initializing hidden layer with size {layer_config[i]}.")
                # Here we add an additional unit in the hidden layers for the bias weight.                
                self.layers.append(Layer([layer_config[i]+1, layer_config[i+1]],
                                         batch_size,
                                         activation=f_sigmoid))  # Comment this line out if you want to use ReLU as activator function
                #                         activation=f_ReLU))  # Comment this line out if you want to use Sigmoid as activator function
        print(f"Initializing output layer with size {layer_config[-1]}.")
        self.layers.append(Layer([layer_config[-1], None],
                                 batch_size,
                                 is_output=True,
        #                         activation=f_softmax))  # Comment this line out if you want to use ReLU as output function
                                 activation=f_ReLU))  # Comment this line out if you want to use Softmax as output function
        print("Done!")

    def forward_propagate(self, data):
        # We need to be sure to add bias values to the input
        self.layers[0].Z = np.append(data, np.ones((data.shape[0], 1)), axis=1)

        for i in range(self.num_layers-1):
            self.layers[i+1].S = self.layers[i].forward_propagate()
        return self.layers[-1].forward_propagate()

    def backpropagate(self, yhat, labels):
        # exit_with_err("FIND ME IN THE CODE, What is computed in the next line of code?\n")
        # Computing error signal for the output layer, output from forward_propagate - the true labels
        self.layers[-1].D = (yhat - labels).T
        """This line is what actually implements the Chain Rule."
        "This row calculates how much each node in the current layer should be 'blamed' (or held responsible) for the final error made by the network."
        "self.layers[i+1].D This is the delta value (the error) from the layer after the current one. Since we are moving backward, we already know the magnitude of the error from layer i+1."
        "self.layers[i].Fp (The derivative of the activation function). ".dot is the dot product."""
        for i in range(self.num_layers-2, 0, -1):
            # We do not calculate deltas for the bias values
            W_nobias = self.layers[i].W[0:-1, :]

            # For loop is adjusting all weights backwards for all activator function outputs
            #exit_with_err("FIND ME IN THE CODE, What does this 'for' loop do?\n")
                
            self.layers[i].D = W_nobias.dot(self.layers[i+1].D) * self.layers[i].Fp

    def update_weights(self, eta):
        for i in range(0, self.num_layers-1):
            W_grad = -eta * (self.layers[i+1].D.dot(self.layers[i].Z)).T
            self.layers[i].W += W_grad

    def evaluate(self, train_data, train_labels, test_data, test_labels, num_epochs=70, eta=0.05, eval_train=False, eval_test=True):

        N_train = len(train_labels) * len(train_labels[0])
        N_test = len(test_labels) * len(test_labels[0])

        print(f"Training for {num_epochs} epochs...")
        for t in range(0, num_epochs):
            out_str = "[{0:4d}] ".format(t)

            for b_data, b_labels in zip(train_data, train_labels):
                output = self.forward_propagate(b_data)
                self.backpropagate(output, b_labels)

                #exit_with_err("FIND ME IN THE CODE, How is the weight update implemented? What is eta?\n")
                self.update_weights(eta=eta)

               
            if eval_train:
                errs = 0
                for b_data, b_labels in zip(train_data, train_labels):
                    output = self.forward_propagate(b_data)
                    yhat = np.argmax(output, axis=1)
                    errs += np.sum(1 - b_labels[np.arange(len(b_labels)), yhat])

                out_str = (f"{out_str} Training error: {float(errs)/N_train:.5f}")

            if eval_test:
                errs = 0
                for b_data, b_labels in zip(test_data, test_labels):
                    output = self.forward_propagate(b_data)
                    yhat = np.argmax(output, axis=1)
                    errs += np.sum(1 - b_labels[np.arange(len(b_labels)), yhat])

                out_str = (f"{out_str} Test error: {float(errs) / N_test:.5f}")

            print(out_str)


In [37]:
def label_to_bit_vector(labels, nbits):
    bit_vector = np.zeros((labels.shape[0], nbits))
    for i in range(labels.shape[0]):
        bit_vector[i, labels[i]] = 1.0

    return bit_vector

In [38]:
def create_batches(data, labels, batch_size, create_bit_vector=False):
    N = data.shape[0]
    print(f"Batch size {batch_size}, the number of examples {N}.")

    if N % batch_size != 0:
        print(f"Warning in create_minibatches(): Batch size {batch_size} does not evenly divide the number of examples {N}.")
    chunked_data = []
    chunked_labels = []
    idx = 0
    while idx + batch_size <= N:
        chunked_data.append(data[idx:idx + batch_size, :])
        if not create_bit_vector:
            chunked_labels.append(labels[idx:idx + batch_size])
        else:
            bit_vector = label_to_bit_vector(labels[idx:idx + batch_size], 10)
            chunked_labels.append(bit_vector)

        idx += batch_size

    return chunked_data, chunked_labels


In [39]:
def prepare_for_backprop(batch_size, train_images, train_labels, valid_images, valid_labels):
    print("Creating data...")
    batched_train_data, batched_train_labels = create_batches(train_images, train_labels,
                                              batch_size, create_bit_vector=True)
    batched_valid_data, batched_valid_labels = create_batches(valid_images, valid_labels,
                                              batch_size, create_bit_vector=True)
    print("Done!")


    return batched_train_data, batched_train_labels,  batched_valid_data, batched_valid_labels



In [40]:
(Xtr, Ltr), (X_test, L_test) = mnist.load_data()

Xtr = Xtr.reshape(60000, 784)
X_test = X_test.reshape(10000, 784)
Xtr = Xtr.astype('float32')
X_test = X_test.astype('float32')
Xtr /= 255
X_test /= 255
print(f"{Xtr.shape[0]} train samples")
print(f"{X_test.shape[0]} test samples")


60000 train samples
10000 test samples


In [45]:
batch_size = 100;

train_data, train_labels, valid_data, valid_labels = prepare_for_backprop(batch_size, Xtr, Ltr, X_test, L_test)

mlp = MultiLayerPerceptron(layer_config=[784, 100, 100, 10], batch_size=batch_size)

# Sigmoid - Don't forget to comment/uncomment the lines up in the class MultiLayerPerceptron __init__ function, which functions you wan't to use
# Different learning rates specified 
#mlp.evaluate(train_data, train_labels, valid_data, valid_labels, eta=0.05, eval_train=True)
#mlp.evaluate(train_data, train_labels, valid_data, valid_labels, eta=0.005, eval_train=True)
#mlp.evaluate(train_data, train_labels, valid_data, valid_labels, eta=0.5, eval_train=True)

# ReLU - learning rate of 0.003 seems comparable to Sigmoid case
mlp.evaluate(train_data, train_labels, valid_data, valid_labels, eta=0.003, eval_train=True)

print("Done:)\n")


Creating data...
Batch size 100, the number of examples 60000.
Batch size 100, the number of examples 10000.
Done!
Initializing input layer with size 784.
Initializing hidden layer with size 100.
Initializing hidden layer with size 100.
Initializing output layer with size 10.
Done!
Training for 70 epochs...
[   0]  Training error: 0.74775 Test error: 0.74280
[   1]  Training error: 0.70712 Test error: 0.70500
[   2]  Training error: 0.69507 Test error: 0.69720
[   3]  Training error: 0.68798 Test error: 0.69110
[   4]  Training error: 0.68407 Test error: 0.68600
[   5]  Training error: 0.68153 Test error: 0.68310
[   6]  Training error: 0.68077 Test error: 0.68360
[   7]  Training error: 0.67973 Test error: 0.68390
[   8]  Training error: 0.67918 Test error: 0.68330
[   9]  Training error: 0.67687 Test error: 0.68070
[  10]  Training error: 0.67188 Test error: 0.67070
[  11]  Training error: 0.66687 Test error: 0.66480
[  12]  Training error: 0.63732 Test error: 0.63640
[  13]  Trainin